In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import shii
import pandas as pd
import geopandas as gpd
from shapely import Point
import matplotlib.pyplot as plt

## Download data from NY OpenData via Socrata API + sodapy

In [ ]:
# Load heatwave dates
hw_dates = pd.read_csv('./heatwaves.csv')['Dates'].values

In [ ]:
heat_list = []
for date in hw_dates:
    heat_incidents = shii.download_heat_incidents(
        app_token=os.getenv("NYC_OPEN_DATA_APP_TOKEN"),
        limit=10000,
        start_timestamp=date,
        end_timestamp=date.replace('T00:00:00', 'T23:59:59')
    )
    heat_list.append(heat_incidents)
heat_df = pd.concat(heat_list, ignore_index=True)

In [ ]:
hydrant_list = []
for date in hw_dates:
    hydrant_incidents = shii.download_311_complaints(
        app_token=os.getenv("NYC_OPEN_DATA_APP_TOKEN"),
        complaint_type="hydrant",
        limit=10000,
        start_timestamp=date,
        end_timestamp=date.replace('T00:00:00', 'T23:59:59')
    )
    hydrant_list.append(hydrant_incidents)
hydrant_df = pd.concat(hydrant_list, ignore_index=True)

In [ ]:
ac_list = []
for date in hw_dates:
    ac_incidents = shii.download_311_complaints(
        app_token=os.getenv("NYC_OPEN_DATA_APP_TOKEN"),
        complaint_type="ac",
        limit=10000,
        start_timestamp=date,
        end_timestamp=date.replace('T00:00:00', 'T23:59:59')
    )
    ac_list.append(ac_incidents)
ac_df = pd.concat(ac_list, ignore_index=True)

In [ ]:
ventilation_list = []
for date in hw_dates:
    ventilation_incidents = shii.download_311_complaints(
        app_token=os.getenv("NYC_OPEN_DATA_APP_TOKEN"),
        complaint_type="ventilation",
        limit=10000,
        start_timestamp=date,
        end_timestamp=date.replace('T00:00:00', 'T23:59:59')
    )
    ventilation_list.append(ventilation_incidents)
ventilation_df = pd.concat(ventilation_list, ignore_index=True)

In [ ]:
power_list = []
for date in hw_dates:
    power_incidents = shii.download_311_complaints(
        app_token=os.getenv("NYC_OPEN_DATA_APP_TOKEN"),
        complaint_type="power",
        limit=10000,
        start_timestamp=date,
        end_timestamp=date.replace('T00:00:00', 'T23:59:59')
    )
    power_list.append(power_incidents)
power_df = pd.concat(power_list, ignore_index=True)


## Get spatial information and plot

In [ ]:
def convert_to_gdf(df, lon_col='longitude', lat_col='latitude', crs='EPSG:4326'):
    """Convert a DataFrame with longitude and latitude columns to a GeoDataFrame."""
    geometry = [Point(xy) for xy in zip(df[lon_col], df[lat_col])]
    gdf = gpd.GeoDataFrame(df, geometry=geometry, crs=crs)
    return gdf

In [ ]:
hydrant_df = convert_to_gdf(hydrant_df)
ac_df = convert_to_gdf(ac_df)
ventilation_df = convert_to_gdf(ventilation_df)
power_df = convert_to_gdf(power_df)
power_df['complaint_type'] = 'power'
ventilation_df['complaint_type'] = 'ventilation'
ac_df['complaint_type'] = 'ac'
hydrant_df['complaint_type'] = 'hydrant'
all_complaints_df = pd.concat([power_df, ventilation_df, ac_df, hydrant_df], ignore_index=True)

In [ ]:
zip_code_gdf = gpd.read_file('https://github.com/fedhere/PUI2015_EC/raw/refs/heads/master/mam1612_EC/nyc-zip-code-tabulation-areas-polygons.geojson')
zip_code_gdf = zip_code_gdf.drop_duplicates('postalCode', keep='first')

In [ ]:
heat_zipcode_counts = heat_df.groupby('zipcode').size()
heat_zipcode_counts.name = 'heat_incident_counts'
zip_code_heat = zip_code_gdf.set_index('postalCode').join(heat_zipcode_counts, how='left')
zip_code_heat['heat_incident_counts'] = zip_code_heat['heat_incident_counts'].fillna(0)

In [ ]:
complaints_zipcode_counts = all_complaints_df.groupby(['incident_zip', 'complaint_type']).size()
complaints_zipcode_counts.name = '311_counts'
complaints_zipcode_counts = complaints_zipcode_counts.reset_index().set_index('incident_zip')
complaints_zipcode_counts = complaints_zipcode_counts.pivot(columns='complaint_type', values='311_counts')
complaints_zipcode_counts = complaints_zipcode_counts.fillna(0)
complaints_zipcode_counts['all'] = complaints_zipcode_counts.sum(axis=1)
zip_code_311 = zip_code_gdf.set_index('postalCode').join(complaints_zipcode_counts, how='left').fillna(0)

In [ ]:
# # Plot 1: Choropleth of heat incidents by ZIP code
fig, ax = plt.subplots(1, 1, figsize=(12, 10))
zip_code_heat.plot(column='heat_incident_counts', cmap='OrRd', linewidth=0.3, edgecolor='gray', legend=True, ax=ax)
ax.set_title(f'Heat EMS Calls by ZIP Code, n = {int(zip_code_heat["heat_incident_counts"].sum())}', fontsize=14, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.show()

complaint_types = ['hydrant','power', 'ac', 'ventilation']
colors = {'power':'tab:red', 'ac':'tab:blue', 'ventilation':'tab:green', 'hydrant':'tab:purple'}
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()
ax = axes[0]
zip_code_heat.plot(column='heat_incident_counts', cmap='OrRd', linewidth=0.3, edgecolor='gray', ax=ax, legend=False)
ax.set_title('Heat Incidents by ZIP Code')
ax.axis('off')

for idx, complaint in enumerate(complaint_types):
    ax = axes[idx+1]
    # base choropleth without legend to keep plots compact
    zip_code_heat.plot(column='heat_incident_counts', cmap='OrRd', linewidth=0.3, edgecolor='gray', legend=False, ax=ax)
    # overlay complaint points (if any) — use small markers
    subset = all_complaints_df.loc[all_complaints_df['complaint_type'] == complaint]
    if not subset.empty:
        subset.plot(ax=ax, markersize=2, alpha=0.3, color=colors[complaint], label=complaint.capitalize())
    ax.set_title(f"{complaint.capitalize()} requests, n = {subset.shape[0]}")
    ax.axis('off')

ax = axes[-1]
ax.set_title('All Request Types')
zip_code_heat.plot(column='heat_incident_counts', cmap='OrRd', linewidth=0.3, edgecolor='gray', legend=False, ax=ax)
for complaint in complaint_types:#
    subset = all_complaints_df.loc[all_complaints_df['complaint_type'] == complaint]
    print(subset.shape)
    if not subset.empty:
        subset.plot(ax=ax, markersize=2, alpha=0.3, color=colors[complaint], label=complaint.capitalize())
        print(colors[complaint])
ax.axis('off')
ax.legend(title='Request Type', loc='upper right')

plt.show()


In [ ]:
zip_code_full = zip_code_heat.join(zip_code_311[['hydrant','ac','power','ventilation','all']], how='inner')

In [ ]:
zip_code_full.plot.scatter('heat_incident_counts', 'all')

In [ ]:
zip_code_full.plot.scatter('heat_incident_counts', 'hydrant')

In [ ]:
zip_code_full.plot.scatter('heat_incident_counts', 'ventilation')

In [ ]:
zip_code_full.plot.scatter('heat_incident_counts', 'power')

## Simple linear modeling

In [ ]:
from scipy import stats
import numpy as np
import matplotlib.pyplot as plt

# Prepare data for regression analysis
complaint_types = ['power', 'ac', 'ventilation', 'hydrant', 'all']

# Dictionary to store regression results
regression_results = {}

# Fit linear models for each complaint type
for complaint in complaint_types:
    # Remove rows with NaN values for this analysis
    clean_data = zip_code_full[[complaint, 'heat_incident_counts']].dropna()
    
    # Extract x and y values
    y = clean_data['heat_incident_counts'].values
    x = clean_data[complaint].values
    
    # Fit linear regression
    slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
    
    # Store results
    regression_results[complaint] = {
        'slope': slope,
        'intercept': intercept,
        'r_value': r_value,
        'r_squared': r_value**2,
        'p_value': p_value,
        'std_err': std_err,
        'n_samples': len(clean_data)
    }
    
    # Print statistics
    print(f"\n{'='*60}")
    print(f"Linear Regression: heat_incident_counts vs {complaint}")
    print(f"{'='*60}")
    print(f"Slope:           {slope:.6f}")
    print(f"Intercept:       {intercept:.6f}")
    print(f"R-value:         {r_value:.6f}")
    print(f"R-squared:       {r_value**2:.6f}")
    print(f"P-value:         {p_value:.6e}")
    print(f"Std Error:       {std_err:.6f}")
    print(f"N samples:       {regression_results[complaint]['n_samples']}")
    
    # Statistical significance
    if p_value < 0.05:
        print("*** Statistically significant (p < 0.05) ***")
    else:
        print("Not statistically significant (p >= 0.05)")

In [ ]:
# Create plots with scatter points and lines of best fit
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, complaint in enumerate(complaint_types):
    ax = axes[idx]
    
    # Prepare clean data
    clean_data = zip_code_full[[complaint, 'heat_incident_counts']].dropna()
    y = clean_data['heat_incident_counts'].values
    x = clean_data[complaint].values
    
    # Get regression parameters
    result = regression_results[complaint]
    
    # Scatter plot
    ax.scatter(x, y, alpha=0.6, s=100)
    
    # Line of best fit
    x_line = np.array([x.min(), x.max()])
    y_line = result['slope'] * x_line + result['intercept']
    ax.plot(x_line, y_line, 'r--', linewidth=2, label=f'y = {result["slope"]:.4f}x + {result["intercept"]:.4f}')
    
    # Labels and title
    ax.set_ylabel('Heat Incident Counts', fontsize=11)
    ax.set_xlabel(f'{complaint.capitalize()} Complaints', fontsize=11)
    ax.set_title(f'{complaint.upper()}\n$R^2$ = {result["r_squared"]:.4f}, p = {result["p_value"]:.2e}', 
                 fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

# Remove the extra subplot
fig.delaxes(axes[5])

plt.tight_layout()
plt.savefig('linear_regression_analysis.png', dpi=100, bbox_inches='tight')
plt.show()

## Model including spatial autocorrelation

Fit spatial lag model

In [ ]:
from libpysal.weights import Queen, weights
from spreg import OLS, ML_Lag
import warnings
warnings.filterwarnings('ignore')

# Create spatial weights matrix based on Queen adjacency (polygons sharing edges or vertices)
# Using zip_code_gdf which has geometry
w_queen = Queen.from_dataframe(zip_code_gdf)
W_mat =weights.W(w_queen.weights)

# Dictionary to store spatial regression results
spatial_regression_results = {}

# Fit both OLS and spatial lag models for each complaint type
for complaint in complaint_types:
    print(f"\n{'='*70}")
    print(f"Spatial Analysis: heat_incident_counts vs {complaint}")
    print(f"{'='*70}")
    
    # Prepare data - ensure the data aligns with spatial weights
    clean_data = zip_code_full[[complaint, 'heat_incident_counts']].dropna()
    
    # Extract y (dependent) and X (independent) variables
    y = clean_data['heat_incident_counts'].values.reshape(-1, 1)
    X = clean_data[[complaint]].values
    
    # Fit OLS model (baseline)
    ols_model = OLS(y, X)
    
    # Fit Spatial Lag model (accounts for spatial autocorrelation in dependent variable)
    try:
        lag_model = ML_Lag(y, X, w_queen, name_x=[complaint], 
                          name_y='heat_incident_counts')
        
        spatial_regression_results[complaint] = {
            'ols': ols_model,
            'lag': lag_model,
            'weights': W_mat
        }
        
        # Print OLS results
        print("\n--- OLS Model (standard regression) ---")
        print(f"Coefficient (X):   {ols_model.betas[1][0]:.6f}")
        print(f"Intercept:         {ols_model.betas[0][0]:.6f}")
        print(f"R-squared:         {ols_model.r2:.6f}")
        print(f"P-value (X):       {ols_model.t_stat[1][1]:.6e}")
        
        # Print Spatial Lag results
        print("\n--- Spatial Lag Model (ML - accounts for spatial autocorrelation) ---")
        print(f"Coefficient (X):   {lag_model.betas[1][0]:.6f}")
        print(f"Intercept:         {lag_model.betas[0][0]:.6f}")
        print(f"Spatial lag (Rho): {lag_model.betas[2][0]:.6f}")
        print(f"R-squared:         {lag_model.pr2:.6f}")

        # Compare models
        print("\n--- Model Comparison ---")
        coef_change = ((lag_model.betas[1][0] - ols_model.betas[1][0]) / ols_model.betas[1][0]) * 100
        print(f"Coefficient change from OLS to Lag: {coef_change:.2f}%")
        
            
    except Exception as e:
        print(f"Error fitting spatial lag model: {e}")
        spatial_regression_results[complaint] = {'ols': ols_model, 'lag': None}

In [ ]:
# Visualize comparison between OLS and Spatial Lag models
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for idx, complaint in enumerate(complaint_types):
    ax = axes[idx]
    
    # Prepare clean data
    clean_data = zip_code_full[[complaint, 'heat_incident_counts']].dropna()
    x = clean_data['heat_incident_counts'].values
    y = clean_data[complaint].values
    
    # Get regression parameters
    result = regression_results[complaint]
    
    # Scatter plot
    ax.scatter(x, y, alpha=0.6, s=100, label='Data points')
    
    # OLS line
    x_line = np.array([x.min(), x.max()])
    y_ols = result['slope'] * x_line + result['intercept']
    ax.plot(x_line, y_ols, 'b--', linewidth=2, label='OLS')
    
    # Spatial Lag line (if available)
    if spatial_regression_results[complaint]['lag'] is not None:
        lag_model = spatial_regression_results[complaint]['lag']
        y_lag = lag_model.betas[1][0] * x_line + lag_model.betas[0][0]
        ax.plot(x_line, y_lag, 'r-', linewidth=2.5, label='Spatial Lag')
    
    # Labels and title
    ax.set_xlabel('Heat Incident Counts', fontsize=11)
    ax.set_ylabel(f'{complaint.capitalize()} Complaints', fontsize=11)
    ax.set_title(f'{complaint.upper()}\n$R^2$ = {result["r_squared"]:.4f}, p = {result["p_value"]:.2e}', 
                 fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

# Remove the extra subplot
fig.delaxes(axes[5])

plt.tight_layout()
plt.savefig('spatial_regression_comparison.png', dpi=100, bbox_inches='tight')
plt.show()